# Day 5: API Fundamentals and FastAPI
**REST principles, HTTP methods, status codes and your first FastAPI app**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi uvicorn httpx requests

# ===== Option B: VS Code / local (run once in the terminal, not here) =====
# python -m venv .venv
# .venv\Scripts\activate        (Windows)   |   source .venv/bin/activate   (macOS/Linux)
# pip install fastapi uvicorn httpx requests
import fastapi
print("FastAPI", fastapi.__version__)

## 2. HTTP from the client side
This cell calls a free public test API. It needs internet access.

In [ ]:
import requests
r = requests.get("https://jsonplaceholder.typicode.com/todos/1")
print("Status:", r.status_code)
print("Content-Type:", r.headers["Content-Type"])
print("Body:", r.json())

r = requests.post("https://jsonplaceholder.typicode.com/posts", json={"title": "Hi", "body": "Day 5", "userId": 1})
print(r.status_code, r.json())

## 3. Your first FastAPI app
`%%writefile` saves the cell as `main.py`. The same file works in VS Code.

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Quotes API", version="1.0")

quotes = {
    1: {"text": "Small steps every day.", "author": "Unknown"},
    2: {"text": "Fall seven times, stand up eight.", "author": "Japanese proverb"},
}

@app.get("/")
def home():
    return {"message": "Welcome to the Quotes API"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/quotes")
def list_quotes():
    return quotes

@app.get("/quotes/{qid}")
def get_quote(qid: int):
    if qid not in quotes:
        raise HTTPException(status_code=404, detail="Quote not found")
    return quotes[qid]

@app.post("/quotes", status_code=201)
def create_quote(text: str, author: str = "Unknown"):
    qid = max(quotes) + 1
    quotes[qid] = {"text": text, "author": author}
    return {"id": qid, **quotes[qid]}

@app.delete("/quotes/{qid}", status_code=204)
def delete_quote(qid: int):
    if qid not in quotes:
        raise HTTPException(status_code=404, detail="Quote not found")
    del quotes[qid]

## 4. Call the API
**Option A (Colab):** `TestClient` calls the app directly, no server needed.

**Option B (VS Code):** run `uvicorn main:app --reload` in the terminal and open http://127.0.0.1:8000/docs

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, main
importlib.reload(main)
from fastapi.testclient import TestClient
client = TestClient(main.app)

# ===== Option B: VS Code with uvicorn running (uncomment to use) =====
# import requests
# class _Client:
#     base = "http://127.0.0.1:8000"
#     def get(self, p, **k): return requests.get(self.base + p, **k)
#     def post(self, p, **k): return requests.post(self.base + p, **k)
#     def delete(self, p, **k): return requests.delete(self.base + p, **k)
# client = _Client()

print(client.get("/").json())
print(client.get("/quotes/2").json())

In [ ]:
r = client.post("/quotes", params={"text": "Keep learning", "author": "Ahmed"})
print("POST:", r.status_code, r.json())

r = client.get("/quotes/99")
print("Missing:", r.status_code, r.json())

r = client.get("/quotes/abc")
print("Wrong type:", r.status_code)

r = client.delete("/quotes/1")
print("DELETE:", r.status_code)

## 5. Status code check
Run these assertions. If nothing prints except the final message, every rule holds.

In [ ]:
assert client.get("/health").status_code == 200
assert client.post("/quotes", params={"text": "Hi"}).status_code == 201
assert client.get("/quotes/999").status_code == 404
assert client.get("/quotes/abc").status_code == 422
print("All status code checks passed")

### TODO 1: Add `GET /quotes/random` and a test for it. Tip: put it above `/quotes/{qid}` so the path matches first.

In [ ]:
# TODO: write your code here

## 6. Optional: run a real server inside Colab
This starts uvicorn in the background and calls it over HTTP.

In [ ]:
# ===== Option A: Google Colab =====
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"])
time.sleep(3)
print(requests.get("http://127.0.0.1:8000/health").json())
server.terminate()

# ===== Option B: VS Code =====
# Just run in the terminal: uvicorn main:app --reload

---
**Yoi shuumatsu wo** (良い週末を): *Have a great weekend*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)